# SpiRob vision pipeline on a robot model (local machine)


## Settings 

In [ ]:
import os, re, glob, json, shutil, subprocess, time

# Settings come from config.env (shared with setup.sh / run.sh). Edit config.env, not this cell.
VISION = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
CFG = {}
for line in open(f"{VISION}/config.env"):
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, v = line.split("=", 1)
        CFG[k.strip()] = os.path.expandvars(v.strip().strip('"'))
SIR_REPO, RUNS, MODEL_TAG = CFG["SIR_REPO"], CFG["RUNS"], CFG["MODEL_TAG"]
NUM_ENVS_TRAIN, NUM_ENVS_EVAL = int(CFG["NUM_ENVS_TRAIN"]), int(CFG["NUM_ENVS_EVAL"])
TEACHER0 = f"{VISION}/checkpoints/teacher_C1_a3.pt"     # starting teacher
STUDENT0 = f"{VISION}/checkpoints/student_c1.pt"        # starting camera student

OUT   = f"{RUNS}/{MODEL_TAG}"
TOOLS = f"{VISION}/tools"
LOGS  = f"{SIR_REPO}/logs/rsl_rl/spirob_minimal"
os.makedirs(OUT, exist_ok=True)
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]   # uv
print(f"model: {MODEL_TAG} | outputs: {OUT}")
for p in (SIR_REPO, TOOLS, TEACHER0, STUDENT0):
    print("found  " if os.path.exists(p) else "MISSING", p)
g = lambda *c: subprocess.run(["git", "-C", SIR_REPO, *c], capture_output=True, text=True).stdout.strip()
print(f"\nsir's repo: branch {g('rev-parse', '--abbrev-ref', 'HEAD')} @ {g('rev-parse', '--short', 'HEAD')}")
print(subprocess.run(["uv", "run", "python", "-c", "import torch; print('GPU:', torch.cuda.get_device_name(0))"],
                     cwd=SIR_REPO, capture_output=True, text=True).stdout or "GPU check failed: run ./setup.sh first")

## Helpers (run once)

In [ ]:
from IPython.display import Image, display

def sh(args, keep=None, env=None):
    """Run a tool from sir's repo folder (so his environment and model are used); stream useful lines."""
    p = subprocess.Popen(["uv", "run", "python", *map(str, args)], cwd=SIR_REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **(env or {})})
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        if keep is None or line.startswith(keep) or "Error" in line or "Traceback" in line:
            print(line.rstrip(), flush=True)
    p.wait()
    if p.returncode != 0:
        print("".join(tail)); raise RuntimeError(f"exit {p.returncode}")

def _num(p): return int(re.findall(r"model_(\d+)\.pt", p)[0])
def ckpts(folder): return sorted(glob.glob(f"{folder}/model_[0-9]*.pt"), key=_num)

def evaluate_teacher(ckpt, task, label, rounds=3):
    sh([f"{TOOLS}/polar_check.py", "--mode", "eval", "--task", task, "--checkpoint", ckpt, "--label", label,
        "--out", OUT, "--num_envs", NUM_ENVS_EVAL, "--rounds", rounds], keep=("round", label, "  ", "saved"))
    png = f"{OUT}/eval_{label}_{task.split('-')[-1]}.png"
    if os.path.isfile(png): display(Image(png, width=650))

KEYS = {"Mean reward": "reward", "Mean episode length": "ep_len",
        "Episode_Termination/success_egg_inside_bucket": "success", "Episode_Termination/egg_fell": "fell",
        "Episode_Termination/nan_state": "crash", "Episode_Termination/time_out": "timeout"}

def finetune(run, task, init_ckpt, iters=500):
    """Continue PPO training of a teacher checkpoint (sir's Stage2FineTuneRunner). One line per iteration."""
    init_dir = f"{LOGS}/cinit-{run}"; os.makedirs(init_dir, exist_ok=True)
    shutil.copy(init_ckpt, f"{init_dir}/model_init.pt")
    dst = f"{OUT}/{run}"; os.makedirs(dst, exist_ok=True)
    def backup():
        for d in glob.glob(f"{LOGS}/*_{run}"):
            for c in ckpts(d): shutil.copy(c, dst)
        return ckpts(dst)
    cmd = ["uv", "run", "python", f"{TOOLS}/train_polar.py", task, "--env.scene.num-envs", str(NUM_ENVS_TRAIN),
           "--agent.max-iterations", str(iters), "--agent.resume", "True", "--agent.load-run", f"cinit-{run}",
           "--agent.load-checkpoint", "model_init.pt", "--agent.run-name", run, "--agent.logger", "tensorboard"]
    print(f"training {run} on {task} from {init_ckpt}", flush=True)
    p = subprocess.Popen(cmd, cwd=SIR_REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    vals, it = {}, None
    def show():
        if it is not None:
            print(f"iter {it:>3} | " + " | ".join(f"{v} {vals.get(v, '-'):>7}" for v in KEYS.values()), flush=True)
    try:
        for line in p.stdout:
            m = re.search(r"Learning iteration (\d+)/", line)
            if m:
                show(); it, vals = int(m.group(1)), {}
                if it % 50 == 1: backup()
                continue
            for k, short in KEYS.items():
                if k + ":" in line: vals[short] = line.split(":")[-1].strip()
            if "Traceback" in line or "Error" in line: print(line.rstrip(), flush=True)
        show(); p.wait()
    finally:
        if p.poll() is None: p.kill()
        cks = backup()
    print(f"{run}: exit {p.returncode} | last checkpoint {cks[-1] if cks else None}")
    return cks[-1] if cks else None

def student(*args):
    sh([f"{TOOLS}/student_dagger.py", "--out", f"{OUT}/student", *args],
       keep=("round", "student", "  ", "saved", "resumed", "camera", "teacher", "[camera]"))

def table():
    """Teacher and student evaluations of this model, side by side."""
    print(subprocess.run(["python3", f"{TOOLS}/summary.py", OUT], capture_output=True, text=True).stdout)
print("helpers ready | outputs ->", OUT)

## 0. Compatibility check 

This checks that the model still gives the teacher **93 inputs** and **2 actions**, that the checkpoints load, that the egg sits stably, and that the cable, keypoint and touch readings respond when the arm moves.

**All PASS:** go on. **Any FAIL:** stop and fix it first. A touch WARN means no contact happened in this short test; if it also appears on the new model, check where the touch sites sit in the XML.

In [ ]:
sh([f"{TOOLS}/check_model.py", "--teacher", TEACHER0, "--student", STUDENT0])

## 1. Teacher with no training ("zero-shot") on the C1 region (125–185 mm, 51–80°)

- **On the current model,** this should give about **93 % nominal, 88 % retention and 50 % sector**, the Colab result. If it does, the local setup is correct.
- **On the new model,** this number shows how big the gap is:
  - about 60 % or more at home: a short fine-tune is enough;
  - about 20–60 %: a normal fine-tune, maybe 2 runs;
  - close to 0 %: fine-tune on Arc-1.5 first (`TASK = "Mjlab-SpiRob-Polar-Arc15"` in Step 2), then C1.

In [ ]:
evaluate_teacher(TEACHER0, "Mjlab-SpiRob-Polar-C1", "teacher_start")

## 2. Teacher fine-tuning 

Each run continues training from the previous checkpoint. **Expect a small dip in the first iterations** while the network adjusts to the new physics.

**Re-run this cell** (with `RUN` set to `"ft2"`, `"ft3"`, …) until the evaluation shows a home area of 75 % or more and a sector close to the old 50 %. The last run's checkpoint becomes `TEACHER`.

In [ ]:
TASK = "Mjlab-SpiRob-Polar-C1"
RUN = "ft1"
prev = sorted(glob.glob(f"{OUT}/ft*/model_[0-9]*.pt"), key=os.path.getmtime)
init = prev[-1] if prev else TEACHER0
ck = finetune(RUN, TASK, init, iters=500)
evaluate_teacher(ck, TASK, RUN)

In [ ]:
# the teacher used from here on: the latest fine-tuned checkpoint, or the starting teacher if none
_ft = sorted(glob.glob(f"{OUT}/ft*/model_[0-9]*.pt"), key=os.path.getmtime)
TEACHER = _ft[-1] if _ft else TEACHER0
print("teacher:", TEACHER)

## 3. Coverage on the whole wide sector (120–200 mm, 30-150°)

This shows whether the new shape opened up areas that were not reachable before: the low angles, the outer edge, the left side.

In [ ]:
evaluate_teacher(TEACHER, "Mjlab-SpiRob-Polar-Wide", "teacher_wide")

## 4. Camera student with no training

Look at the **camera egg-position error** in the output:
- **About 4–6 mm:** the camera part still works with the new arm appearance.
- **Much larger:** the new look confuses it. Step 5 fixes that by retraining.

In [ ]:
student("--mode", "check")
display(Image(f"{OUT}/student/camera_check.png", width=900))

In [ ]:
student("--mode", "eval", "--task", "Mjlab-SpiRob-Polar-C1", "--student", STUDENT0, "--label", "student_start",
        "--num_envs", NUM_ENVS_EVAL // 2, "--eval_rounds", 3)

## 5. Camera student DAgger, starting from the current student

The student copies `TEACHER` (the fine-tuned teacher for this model). This is 8 rounds; the teacher's share goes 50 → 30 → 20 → 10 → 0 %.

If it stops part-way (for example out of GPU memory), use `student_last.pt`: it's saved after every round.

In [ ]:
student("--mode", "train", "--task", "Mjlab-SpiRob-Polar-C1", "--teacher", TEACHER, "--init", STUDENT0,
        "--num_envs", NUM_ENVS_TRAIN // 2, "--rounds", 8, "--betas", "0.5,0.3,0.2,0.1,0")

## 6. Student evaluation and camera tests

- **Normal:** the student's real performance.
- **Black image:** the camera removed.
- **Far-swap:** each robot is shown the image from the robot whose egg is farthest away.

If the student uses the camera, both broken-image tests must score clearly below normal.

In [ ]:
STUDENT = f"{OUT}/student/student_last.pt"
for abl in ("none", "black", "farswap"):
    student("--mode", "eval", "--task", "Mjlab-SpiRob-Polar-C1", "--student", STUDENT, "--label", "student_new",
            "--ablate", abl, "--num_envs", NUM_ENVS_EVAL // 2, "--eval_rounds", 3)

## 7. Summary of this model

In [ ]:
table()